In [1]:
import pandas as pd


In [2]:
file_path1=r'D:\SQL learning 2\airbnb_project\Bristol\download\csv_file\Bristol_listings.csv'
listings = pd.read_csv(file_path1, encoding="latin1")

In [3]:
print(listings.columns)

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'ca

In [4]:
print(listings.property_type)  # (rows, columns)

0       Private room in bed and breakfast
1                    Private room in home
2                      Entire rental unit
3                      Entire rental unit
4                    Private room in home
                      ...                
2767                          Entire home
2768                   Entire rental unit
2769                         Entire condo
2770                         Entire place
2771                   Entire rental unit
Name: property_type, Length: 2772, dtype: object


In [5]:
# assume your dataframe is called df
# count the occurrences of each property_type
counts = listings['property_type'].value_counts()
print(counts)


property_type
Entire rental unit                   846
Entire home                          586
Private room in home                 468
Entire condo                         253
Entire serviced apartment            125
Private room in rental unit          108
Private room in townhouse             90
Private room in condo                 65
Entire townhouse                      61
Private room in bed and breakfast     25
Entire guesthouse                     23
Tiny home                             21
Entire guest suite                    15
Room in hotel                         14
Entire loft                            8
Private room                           7
Entire cottage                         6
Room in serviced apartment             6
Entire place                           6
Private room in guest suite            5
Private room in tiny home              5
Entire villa                           4
Entire cabin                           3
Private room in casa particular        2
Tr

In [6]:
# keep only property_types that appear at least 3 times
valid_types = counts[counts >= 15].index

# filter the dataframe
listings_filtered = listings[listings['property_type'].isin(valid_types)]

In [7]:
print(listings_filtered.property_type)  # (rows, columns)

0       Private room in bed and breakfast
1                    Private room in home
2                      Entire rental unit
3                      Entire rental unit
4                    Private room in home
                      ...                
2766                   Entire guest suite
2767                          Entire home
2768                   Entire rental unit
2769                         Entire condo
2771                   Entire rental unit
Name: property_type, Length: 2686, dtype: object


In [8]:
# Absolute booked nights last year
listings_filtered["abs_occupancy"] = listings_filtered["estimated_occupancy_l365d"]

# Proxy occupancy rate (% of 365)
listings_filtered["proxy_occupancy_pct"] = listings_filtered["estimated_occupancy_l365d"] / 365 * 100

C:\Users\tsang\AppData\Local\Temp\ipykernel_14940\2606022214.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  listings_filtered["abs_occupancy"] = listings_filtered["estimated_occupancy_l365d"]
C:\Users\tsang\AppData\Local\Temp\ipykernel_14940\2606022214.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  listings_filtered["proxy_occupancy_pct"] = listings_filtered["estimated_occupancy_l365d"] / 365 * 100


In [9]:
#Flag possible seasonal / part-time listings
#If abs_occupancy is high, but proxy_occupancy_pct looks suspiciously low,
#it suggests the host didn’t open all 365 days.
listings_filtered["likely_seasonal"] = (
    (listings_filtered["abs_occupancy"] > 90) &   # more than ~3 months booked
    (listings_filtered["proxy_occupancy_pct"] < 40)  # but proxy looks low
)


C:\Users\tsang\AppData\Local\Temp\ipykernel_14940\2941355898.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  listings_filtered["likely_seasonal"] = (


In [10]:
bins = [0, 1, 3, 5, 8, 12, 20]
labels = ["1", "2-3", "4-5", "6-8", "9-12", "13-20"]

listings_filtered["accommodates_band"] = pd.cut(
    listings_filtered["accommodates"],
    bins=bins,
    labels=labels,
    right=True,   # include right edge of interval
    include_lowest=True
)

print(listings_filtered[["accommodates", "accommodates_band"]].head(20))


    accommodates accommodates_band
0              2               2-3
1              2               2-3
2              2               2-3
3              7               6-8
4              4               4-5
5              3               2-3
6              1                 1
7              2               2-3
8              1                 1
9              5               4-5
13             2               2-3
14             4               4-5
15             2               2-3
16             2               2-3
17             3               2-3
19             2               2-3
20             1                 1
21             2               2-3
23            12              9-12
24             1                 1


C:\Users\tsang\AppData\Local\Temp\ipykernel_14940\1039618955.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  listings_filtered["accommodates_band"] = pd.cut(


In [11]:
groups = listings_filtered.groupby(["neighbourhood_cleansed", "room_type", "accommodates_band"])
unique_groups = listings_filtered[["neighbourhood_cleansed", "room_type", "accommodates"]].drop_duplicates()

print(groups.ngroups)
print(len(unique_groups))

241
387


In [12]:
group_sizes = groups.size().reset_index(name="count")
print(group_sizes["count"].describe())  # summary stats
#print(group_sizes.sort_values("count").head(20))  # smallest groups


count    408.000000
mean       6.583333
std       12.851559
min        0.000000
25%        0.000000
50%        1.000000
75%        7.000000
max      109.000000
Name: count, dtype: float64


In [13]:
print(listings_filtered[["neighbourhood_cleansed", "room_type", "accommodates_band"]].isna().sum())


neighbourhood_cleansed    0
room_type                 0
accommodates_band         0
dtype: int64


# See the unique values per column
print(listings_filtered["accommodates"].unique())
print(listings_filtered["accommodates_band"].unique())
print(listings_filtered["room_type"].nunique())
print(listings_filtered["property_type"].nunique())
print(listings_filtered["neighbourhood_cleansed"].nunique())



In [14]:
# Remove $ and commas, convert to float
listings_filtered["price"] = (
    listings_filtered["price"]
    .replace('[\$,]', '', regex=True)  # strip $ and commas
    .astype(float)
)


C:\Users\tsang\AppData\Local\Temp\ipykernel_14940\2988999277.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  listings_filtered["price"] = (


In [15]:
benchmarks = groups.agg(
    avg_price=("price", "mean"),
    median_abs_occ=("abs_occupancy", "median"),
    median_proxy_occ=("proxy_occupancy_pct", "median")
).reset_index()

# Merge benchmarks back into the listings
listings_filtered = listings_filtered.merge(
    benchmarks,
    on=["neighbourhood_cleansed", "room_type", "accommodates_band"],   # match the same group keys you used
    how="left"
)

In [16]:
listings_filtered["overpriced"] = (
    (listings_filtered["price"] > listings_filtered["avg_price"]) &
    (listings_filtered["abs_occupancy"] < listings_filtered["median_abs_occ"])
)

listings_filtered["underpriced"] = (
    (listings_filtered["price"] < listings_filtered["avg_price"]) &
    (listings_filtered["abs_occupancy"] > listings_filtered["median_abs_occ"])
)


In [17]:
print(listings_filtered.overpriced.value_counts())
print(listings_filtered.underpriced.value_counts())

overpriced
False    2298
True      388
Name: count, dtype: int64
underpriced
False    1941
True      745
Name: count, dtype: int64


In [18]:
# Filter listings where overpriced is True
overpriced_ids = listings_filtered[listings_filtered["overpriced"] == True]["id"]

# Print them
print(overpriced_ids.tolist())   # prints as a Python list

underpriced_ids = listings_filtered[listings_filtered["underpriced"] == True]["id"]
print(underpriced_ids.tolist())   # prints as a Python list

# Suppose you have columns in your DataFrame indicating overpriced/underpriced status
# Example: listings_filtered["overpriced"] and listings_filtered["underpriced"]

# Create a new DataFrame with only the relevant columns
df_export = listings_filtered[["id","overpriced", "underpriced"]]

# Export to Excel
df_export.to_excel(r"D:\SQL learning 2\airbnb_project\Bristol\bnb_pricing_flags.xlsx", index=False)



[45434492.0, 7.97137e+17, 28635166.0, 1.01675e+18, 1.07023e+18, 1.30478e+18, 22772461.0, 14710856.0, 1.37448e+18, 1.22824e+18, 6.33734e+17, 1.31247e+18, 9.08729e+17, 1.36373e+18, 44426766.0, 9.54256e+17, 8.33251e+17, 46183832.0, 11989628.0, 1.27475e+18, 1.22246e+18, 1.05072e+18, 7.94949e+17, 1.35068e+18, 1.31318e+18, 51678453.0, 1.26453e+18, 21366101.0, 9.84023e+17, 6.2532e+17, 8.37026e+17, 4315598.0, 1.23982e+18, 15223307.0, 53675108.0, 9.83963e+17, 19043909.0, 9.27992e+17, 29432972.0, 1.27113e+18, 30401303.0, 1.09368e+18, 9.89199e+17, 1.25808e+18, 1.01587e+18, 1.06236e+18, 23320983.0, 9.67418e+17, 33764113.0, 25993302.0, 1.30361e+18, 50281651.0, 1.34541e+18, 8.57159e+17, 53621600.0, 9.29832e+17, 15741326.0, 45409857.0, 1.32306e+18, 1.37451e+18, 1.29035e+18, 39545681.0, 6.1953e+17, 51277087.0, 6.9345e+17, 1.3327e+18, 30192991.0, 1.04639e+18, 1.21581e+18, 1.37457e+18, 5566863.0, 52990052.0, 52415688.0, 5.55751e+17, 25967696.0, 39914485.0, 32780334.0, 24317219.0, 6.89528e+17, 33314388.0

In [19]:
# Filter IDs where overpriced is True
overpriced_ids = listings_filtered.loc[listings_filtered["overpriced"], "id"]

# Print them
print(overpriced_ids.tolist())

overpriced_ids.to_frame(name="id").to_excel(
    r"D:\SQL learning 2\airbnb_project\Bristol\overpriced_ids.xlsx",
    index=False
)

[45434492.0, 7.97137e+17, 28635166.0, 1.01675e+18, 1.07023e+18, 1.30478e+18, 22772461.0, 14710856.0, 1.37448e+18, 1.22824e+18, 6.33734e+17, 1.31247e+18, 9.08729e+17, 1.36373e+18, 44426766.0, 9.54256e+17, 8.33251e+17, 46183832.0, 11989628.0, 1.27475e+18, 1.22246e+18, 1.05072e+18, 7.94949e+17, 1.35068e+18, 1.31318e+18, 51678453.0, 1.26453e+18, 21366101.0, 9.84023e+17, 6.2532e+17, 8.37026e+17, 4315598.0, 1.23982e+18, 15223307.0, 53675108.0, 9.83963e+17, 19043909.0, 9.27992e+17, 29432972.0, 1.27113e+18, 30401303.0, 1.09368e+18, 9.89199e+17, 1.25808e+18, 1.01587e+18, 1.06236e+18, 23320983.0, 9.67418e+17, 33764113.0, 25993302.0, 1.30361e+18, 50281651.0, 1.34541e+18, 8.57159e+17, 53621600.0, 9.29832e+17, 15741326.0, 45409857.0, 1.32306e+18, 1.37451e+18, 1.29035e+18, 39545681.0, 6.1953e+17, 51277087.0, 6.9345e+17, 1.3327e+18, 30192991.0, 1.04639e+18, 1.21581e+18, 1.37457e+18, 5566863.0, 52990052.0, 52415688.0, 5.55751e+17, 25967696.0, 39914485.0, 32780334.0, 24317219.0, 6.89528e+17, 33314388.0

In [20]:
# Get overpriced IDs
overpriced_ids = listings_filtered.loc[listings_filtered["overpriced"], "id"]

# Get underpriced IDs
underpriced_ids = listings_filtered.loc[listings_filtered["underpriced"], "id"]

# Convert to DataFrames and reset index
df_overpriced = overpriced_ids.reset_index(drop=True).to_frame(name="overpriced")
df_underpriced = underpriced_ids.reset_index(drop=True).to_frame(name="underpriced")

# Combine into a single DataFrame
df_combined = pd.concat([df_overpriced, df_underpriced], axis=1)

# Export to Excel
df_combined.to_excel(
    r"D:\SQL learning 2\airbnb_project\Bristol\bnb_pricing_candidates_refined.xlsx",
    index=False
)
